# 3.6 Calibration du modèle

# Q1

In [12]:
import numpy as np
from scipy.stats import norm

t       = 0.0
delta   = 1.0
lambda_ = 0.05
N       = 1.0
Ti      = 6.0
Ti_fix  = Ti - delta

B_Ti     = 0.830609 # valeur reprise sur la courbe zero
B_Ti_fix = 0.863028

L_fwd = (1/delta) * (B_Ti_fix / B_Ti - 1)
K     = L_fwd

price_market = 0.007137

print(f"L_ATM           = {L_fwd:.4%}")
print(f"Prix marché ATM = {price_market:.4%}")

def black_caplet_HW(sigma_i, L, K, delta, B_Ti, Ti, t, N=1):
    Z_t     = 1 + delta * L
    K_tilde = 1 + delta * K
    Ti_fix  = Ti - delta
    sqrtT   = np.sqrt(Ti_fix - t)
    d1 = (np.log(Z_t / K_tilde) + 0.5 * sigma_i**2 * (Ti_fix - t)) / (sigma_i * sqrtT)
    d2 = sigma_i * sqrtT - d1
    return N * delta * B_Ti * (Z_t * norm.cdf(d1) - K_tilde * norm.cdf(-d2))

def dichotomie(price_cible, L, K, delta, B_Ti, Ti, t, N=1,
               tol=1e-6, s_low=1e-4, s_high=1.0):
    f_low  = black_caplet_HW(s_low,  L, K, delta, B_Ti, Ti, t, N)
    f_high = black_caplet_HW(s_high, L, K, delta, B_Ti, Ti, t, N)
    if f_low  > price_cible: return s_low
    if f_high < price_cible: return s_high
    while s_high - s_low > tol:
        s_mid = (s_low + s_high) / 2
        f_low = black_caplet_HW(s_low, L, K, delta, B_Ti, Ti, t, N)
        f_mid = black_caplet_HW(s_mid, L, K, delta, B_Ti, Ti, t, N)
        if (f_low - price_cible) * (f_mid - price_cible) > 0:
            s_low = s_mid
        else:
            s_high = s_mid
    return (s_low + s_high) / 2

sigma_i = dichotomie(price_market, L_fwd, K, delta, B_Ti, Ti, t, N)
print(f"Volatilité implicite   = {sigma_i*100:.4f} %")

beta_sq   = ((1 - np.exp(-lambda_ * delta)) / lambda_)**2
phi_coeff = (1 - np.exp(-2 * lambda_ * Ti_fix)) / (2 * lambda_)
sigma     = np.sqrt(sigma_i**2 * Ti_fix / (beta_sq * phi_coeff))

print(f"Volatilité instantanée = {sigma*100:.4f} %")

L_ATM           = 3.9030%
Prix marché ATM = 0.7137%
Volatilité implicite   = 0.9270 %
Volatilité instantanée = 1.0713 %


# Q2

In [13]:
# Strikes en bps
strikes_bps = [-100, -50, -25, 0, 25, 50, 100]

print(f"{'Strike (bps)':<15} {'Strike (%)':<15} {'Prix modèle HW':<15}")
print("-" * 45)

for s_bps in strikes_bps:
    K_s = L_fwd + s_bps / 10000
    price_HW = black_caplet_HW(sigma_i, L_fwd, K_s, delta, B_Ti, Ti, t, N)
    print(f"{s_bps:<15} {K_s:.4%}        {price_HW:.4%}")

Strike (bps)    Strike (%)      Prix modèle HW 
---------------------------------------------
-100            2.9030%        1.2015%
-50             3.4030%        0.9388%
-25             3.6530%        0.8214%
0               3.9030%        0.7137%
25              4.1530%        0.6155%
50              4.4030%        0.5268%
100             4.9030%        0.3770%
